# Which sentences drive the prediction? — occlusion attribution

**Why not just classify each sentence.** We tried that first. On the Cheers
transcript the model returned "You should take two capsules daily" as Fear
Appeals at 99%, "it is the rate-limiting step or the bottleneck" as Scarcity at
97%, and an enzyme description as Social Proof at 83% — while the sentence that
*is* a fear appeal came back as Scarcity. Every sentence in isolation is far
outside the retail ad copy the model was fine-tuned on, so it falls back on
lexical association and does so at very high confidence. Confident nonsense.

**What this does instead.** Classify the whole ad once — the in-distribution
call the model was actually trained for. Then remove one sentence at a time and
reclassify. The drop in confidence for the original label measures how much
that sentence was carrying the prediction.

The claim changes from *"this sentence is Fear Appeals"* (which the model cannot
support) to *"removing this sentence drops Fear Appeals from 38% to 12%"* — a
statement about model behaviour, which is exactly what it can support.

This is standard occlusion / leave-one-out attribution. Cost is N+1 forward
passes; 33 sentences is a few seconds on CPU.

**Two limits to state in the report.** Attribution over a weak prediction is
itself weak — if the whole-ad confidence is 38%, the drops will be small and
noisy. And occlusion assumes sentences contribute independently, which they do
not.

## Setup

In [1]:
import re, sys
from pathlib import Path

REPO = Path(r"C:\ETH\deeplabv3plus\DeepLabV3Plus-Pytorch\Group-6-Final-Project")
sys.path.insert(0, str(REPO / "app"))

import predict, tactics

print("model ready:", predict.is_ready())
print("guard:", tactics.LOW_CONFIDENCE)

model ready: True
guard: 0.6


In [2]:
TIMESTAMP = re.compile(r"^\s*\d{1,2}:\d{2}:\d{2}\s*$", re.M)
FOOTER    = re.compile(r"^\s*>\s*Generated by.*$", re.M | re.I)
UI_NOISE  = re.compile(r"\b(?:Follow|Add comment|Show more|Like|Share|Sponsored)\b", re.I)

def clean_transcript(text):
    text = FOOTER.sub("", text)
    text = TIMESTAMP.sub(" ", text)
    text = UI_NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

SENT_END = re.compile(r'(?<=[.!?])\s+(?=[A-Z"\'(])')

def split_sentences(text, min_words=4):
    """Split into sentences, keeping (start, end) offsets into `text`.

    Fragments under min_words are merged backwards — 'Oh, look at that.' cannot
    carry a tactic alone and occluding it would measure nothing.
    """
    spans, start = [], 0
    for m in SENT_END.finditer(text):
        seg = text[start:m.start()].strip()
        if seg:
            spans.append((start, m.start(), seg))
        start = m.end()
    tail = text[start:].strip()
    if tail:
        spans.append((start, len(text), tail))

    merged = []
    for s, e, t in spans:
        if merged and len(t.split()) < min_words:
            ps, _, pt = merged[-1]
            merged[-1] = (ps, e, (pt + " " + t).strip())
        else:
            merged.append((s, e, t))
    return merged

print("ready")

ready


## Load the ad

In [3]:
SOURCE = r"C:\Users\shash\Downloads\Video by thecheersceo.txt"

body = clean_transcript(Path(SOURCE).read_text(encoding="utf-8"))
sentences = split_sentences(body)

print(f"{len(body.split())} words -> {len(sentences)} sentences")

584 words -> 33 sentences


## Baseline: the whole-ad prediction

Everything below is measured relative to this. If the baseline confidence is
low, say so — attribution over an uncertain prediction is uncertain too.

In [4]:
base = predict.predict(body)
BASE_LABEL = base["label"]
BASE_CONF  = base["confidence"]

print(f"whole ad -> {BASE_LABEL} at {BASE_CONF:.1%}")
print(f"windows: {base.get('windows', 1)}, subwords: {base.get('token_count', '?')}\n")
for d in base["distribution"]:
    bar = "#" * int(d["confidence"] * 40)
    print(f"  {d['label']:<24}{d['confidence']:>6.1%}  {bar}")

if BASE_CONF < tactics.LOW_CONFIDENCE:
    print(f"\n  NOTE: baseline is below the {tactics.LOW_CONFIDENCE} guard.")
    print("  Attribution over a weak prediction is weak. Report it as such.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (825 > 512). Running this sequence through the model will result in indexing errors


whole ad -> Fear Appeals at 42.5%
windows: 9, subwords: 827

  Fear Appeals             42.5%  ################
  Authority Manipulation   21.9%  ########
  Scarcity                 14.2%  #####
  Social Proof             11.9%  ####
  Exaggerated Claims        8.4%  ###
  Urgency                   0.4%  
  FOMO                      0.3%  
  Neutral                   0.2%  

  NOTE: baseline is below the 0.6 guard.
  Attribution over a weak prediction is weak. Report it as such.


## Occlusion

For each sentence: remove it, reclassify what remains, and record how far the
baseline label's confidence falls. Positive drop = the sentence was supporting
the prediction. Negative = removing it made the model *more* confident, so that
sentence was arguing against the label.

In [5]:
def confidence_for(text, label):
    """Confidence assigned to `label`, whether or not it is the argmax."""
    if not text.strip():
        return 0.0
    p = predict.predict(text)
    for d in p["distribution"]:
        if d["label"] == label:
            return d["confidence"]
    return 0.0

def occlude(sentences, base_label, base_conf):
    results = []
    n = len(sentences)
    for i, (start, end, sent) in enumerate(sentences):
        remaining = " ".join(t for j, (_, _, t) in enumerate(sentences) if j != i)
        conf = confidence_for(remaining, base_label)
        results.append({
            "index": i, "start": start, "end": end, "text": sent,
            "without": conf,
            "drop": base_conf - conf,
            "phrases": {k: v for k, v in tactics.find_phrases(sent).items() if v},
        })
        print(f"  {i+1:>3}/{n}", end="\r")
    print(" " * 20, end="\r")
    return results

occ = occlude(sentences, BASE_LABEL, BASE_CONF)
print(f"scored {len(occ)} sentences against baseline {BASE_LABEL} {BASE_CONF:.1%}")

scored 33 sentences against baseline Fear Appeals 42.5%


## What the model was reacting to

In [6]:
ranked = sorted(occ, key=lambda r: -r["drop"])

print(f"Baseline: {BASE_LABEL} at {BASE_CONF:.1%}\n")
print(f"{'drop':>7}{'without':>9}   sentence")
print("-" * 78)
for r in ranked:
    flag = "*" if r["drop"] > 0.02 else " "
    ph = f"  [{', '.join(r['phrases'])}]" if r["phrases"] else ""
    print(f"{flag}{r['drop']:>6.1%}{r['without']:>9.1%}   {r['text'][:52]}{ph}")

print("\n* = removing this sentence measurably weakens the prediction")

Baseline: Fear Appeals at 42.5%

   drop  without   sentence
------------------------------------------------------------------------------
*  7.3%    35.2%   You should take two capsules of Cheers Protect daily
*  6.2%    36.3%   Not only will it reduce acetaldehyde when taken befo
*  6.2%    36.3%   And acetaldehyde is twenty times more toxic than the
*  5.4%    37.1%   So now let's have another drink of alcohol, which is
*  5.2%    37.2%   And of course, this gets worse when you turn forty, 
*  5.2%    37.3%   Your body makes glutathione through glycine, glutama
*  5.0%    37.5%   So the first reason is if you take Cheers Protect da
*  4.8%    37.6%   Oh, look at that. It's not magic.
*  4.8%    37.7%   And unfortunately, for most people starting at aroun
*  3.8%    38.7%   It becomes twenty times more toxic when in your live
*  3.7%    38.8%   Glutathione is your body's master antioxidant, and u
*  3.2%    39.2%   Oh, look at that. It's not magic.
*  3.1%    39.4%   And let's see w

## The panel

Only sentences whose removal changes the prediction meaningfully are shown. The
threshold is a share of the baseline confidence, not an absolute — a 2-point
drop means something different against a 38% baseline than against a 90% one.

In [7]:
RELATIVE_THRESHOLD = 0.10   # sentence must account for >=10% of the confidence

cutoff = BASE_CONF * RELATIVE_THRESHOLD
drivers = [r for r in ranked if r["drop"] >= cutoff]

print(f"threshold: drop >= {cutoff:.1%}  ({RELATIVE_THRESHOLD:.0%} of baseline)\n")

if not drivers:
    print("No single sentence carries the prediction.")
    print("The signal is spread across the ad rather than concentrated, which is")
    print("itself worth reporting — occlusion cannot localise it.")
else:
    display = tactics.display_name(BASE_LABEL)
    print(f"This ad reads as {display}. The wording doing that work:\n")
    for r in drivers:
        print(f'  "{r["text"][:150]}{"..." if len(r["text"])>150 else ""}"')
        print(f"     without it, {display} falls {BASE_CONF:.0%} -> {r['without']:.0%}")
        ph = [p["text"] for lst in r["phrases"].values() for p in lst]
        if ph:
            print(f"     trigger phrases: {ph}")
        print()
    print(f"  {tactics.explain(BASE_LABEL, [p for r in drivers for lst in r['phrases'].values() for p in lst])}")

threshold: drop >= 4.2%  (10% of baseline)

This ad reads as Fear. The wording doing that work:

  "You should take two capsules of Cheers Protect daily."
     without it, Fear falls 42% -> 35%

  "Not only will it reduce acetaldehyde when taken before or after consuming alcohol, or ideally both, but it is actually the precursor to glutathione."
     without it, Fear falls 42% -> 36%

  "And acetaldehyde is twenty times more toxic than the alcohol it was converted from."
     without it, Fear falls 42% -> 36%

  "So now let's have another drink of alcohol, which is gonna convert into the acetaldehyde, but this time we took Cheers Protect beforehand and dosed ou..."
     without it, Fear falls 42% -> 37%

  "And of course, this gets worse when you turn forty, and even worse when you turn fifty, and even worse as you turn sixty, seventy, eighty."
     without it, Fear falls 42% -> 37%

  "Your body makes glutathione through glycine, glutamate, and cysteine."
     without it, Fear falls 4

## Sensitivity to the threshold

Sweep it. Too low and every sentence is a driver; too high and none are. Read
the sentences at each level rather than picking a number in the abstract.

In [8]:
for t in [0.05, 0.10, 0.20, 0.30, 0.50]:
    c = BASE_CONF * t
    n = sum(1 for r in ranked if r["drop"] >= c)
    print(f"{t:>5.0%} of baseline (drop >= {c:>5.1%}) -> {n:>2} driver sentence(s)")

   5% of baseline (drop >=  2.1%) -> 16 driver sentence(s)
  10% of baseline (drop >=  4.2%) ->  9 driver sentence(s)
  20% of baseline (drop >=  8.5%) ->  0 driver sentence(s)
  30% of baseline (drop >= 12.7%) ->  0 driver sentence(s)
  50% of baseline (drop >= 21.2%) ->  0 driver sentence(s)


## Attribution for the other tactics

The baseline label is not the only tactic present. Repeating occlusion against
a second label shows where *it* lives — which is how a multi-tactic ad gets a
multi-section panel without asking the model to classify fragments.

In [9]:
SECOND = base["distribution"][1]["label"]
SECOND_CONF = base["distribution"][1]["confidence"]

print(f"second tactic: {SECOND} at {SECOND_CONF:.1%}\n")

occ2 = occlude(sentences, SECOND, SECOND_CONF)
ranked2 = sorted(occ2, key=lambda r: -r["drop"])

cut2 = SECOND_CONF * RELATIVE_THRESHOLD
drivers2 = [r for r in ranked2 if r["drop"] >= cut2]

if drivers2:
    print(f"{tactics.display_name(SECOND)} is carried by:\n")
    for r in drivers2[:5]:
        print(f'  "{r["text"][:120]}..."')
        print(f"     {SECOND_CONF:.0%} -> {r['without']:.0%}\n")
else:
    print("No sentence localises the second tactic.")

second tactic: Authority Manipulation at 21.9%

Authority is carried by:

  "It looks it, but it's science...."
     22% -> 10%

  "Everyone always says that alcohol is toxic, and they are wrong, and I'm gonna explain why with science and what you can ..."
     22% -> 13%

  "Oh, look at that. It's not magic...."
     22% -> 16%

  "And let's see what happens when we pour in the acetaldehyde...."
     22% -> 17%

  "You should take two capsules of Cheers Protect daily...."
     22% -> 18%



## Notes for the report

- Occlusion measures **model sensitivity**, not ground truth about the ad. The
  honest phrasing is "the model relied on this wording", not "this sentence is
  a fear appeal".
- Compare against the per-sentence classification approach: that produced
  confident, wrong labels on out-of-distribution fragments. This produces
  weaker but defensible claims. Both results belong in the error analysis — the
  first is the more interesting failure.
- Attribution over a low-confidence baseline inherits that uncertainty.
- Occlusion assumes independent contributions; overlapping or redundant
  sentences will each show a small drop even if together they carry the
  prediction.
- The underlying fix is training data that contains long-form and spoken ad
  copy, so neither whole-ad nor sentence-level inference is out of
  distribution. That is dataset work, and belongs in future work.